# 🔍 Phase 3: Topic Discovery — LPDP

---

## 🎯 Tujuan Notebook
Mengelompokkan **artikel LPDP** ke dalam topik-topik utama menggunakan **BERTopic** berbasis contextual embeddings,
untuk memahami tema dominan sebelum analisis sentimen dilakukan.

### Alur Proses:
| Langkah | Deskripsi |
|---------|----------|
| 1️⃣ Load Dataset | Muat `dataset_lpdp_konten_raw.csv` (1.038 artikel dengan konten) |
| 2️⃣ Preprocessing | Normalisasi teks minimal (lowercase, hapus URL & noise) |
| 3️⃣ Embeddings | Encode artikel ke vektor via `paraphrase-multilingual-MiniLM-L12-v2` |
| 4️⃣ BERTopic Training | Fit BERTopic → cluster ke 4 topik utama |
| 5️⃣ Analisis Topik | Interpretasi label topik + distribusi per sentimen |
| 6️⃣ Visualisasi | Bar chart distribusi topik + heatmap sentimen × topik |
| 7️⃣ Export | Simpan topic assignment ke `dataset_lpdp_topics.csv` |

> 🧠 **Kenapa BERTopic?** BERTopic menggunakan contextual embeddings (transformer) sehingga
> topik yang dihasilkan lebih koheren dan interpretable dibanding LDA (bag-of-words).

---

**Tim:** Kelompok 5 — Iqbal, Amel, Celine, Salwa, Nida  
**PIC Phase 3:** Celine  
**Input:** `dataset_lpdp_konten_raw.csv` — 1.038 artikel dengan konten lengkap (Positive: 385 · Neutral: 342 · Negative: 311)  
**Output:** `dataset_lpdp_topics.csv` — artikel + kolom `Topic` & `Topic_Prob`

## 📦 Import Libraries

Library yang digunakan:
- **bertopic**: Topic modeling berbasis contextual embeddings
- **sentence-transformers**: Model multilingual untuk encode teks ke vektor
- **pandas / numpy**: Manipulasi DataFrame dan array
- **matplotlib / seaborn**: Visualisasi distribusi topik dan sentimen
- **re / os**: Preprocessing teks dan operasi file

> ⚠️ **SYARAT PYTHON**: `bertopic` membutuhkan **Python 3.11 atau 3.12**.  
> Python 3.13+ belum didukung karena dependency `hdbscan` belum punya wheel-nya.  
> **Jika kamu menggunakan Python 3.13+**, jalankan notebook ini di:
> - 🔗 [Google Colab](https://colab.research.google.com/) (gratis, Python 3.11 default), atau
> - 💻 Virtual environment Python 3.11/3.12 lokal
>
> Cek versi Python aktif: `import sys; print(sys.version)`

In [3]:
import sys

print(f"Python version : {sys.version}")
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

major, minor = sys.version_info.major, sys.version_info.minor
if (major, minor) >= (3, 13):
    print()
    print("⛔ Python 3.13+ TIDAK KOMPATIBEL dengan bertopic (hdbscan belum ada wheel-nya).")
    print("   Jalankan notebook ini di Google Colab atau env Python 3.11/3.12.")
    print("   https://colab.research.google.com/")
elif (major, minor) in [(3, 11), (3, 12)]:
    print("✅ Python kompatibel — lanjutkan ke cell berikutnya.")
else:
    print(f"⚠️  Python {major}.{minor} belum diuji. Disarankan Python 3.11 atau 3.12.")

Python version : 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Python 3.14.3

⛔ Python 3.13+ TIDAK KOMPATIBEL dengan bertopic (hdbscan belum ada wheel-nya).
   Jalankan notebook ini di Google Colab atau env Python 3.11/3.12.
   https://colab.research.google.com/


In [4]:
# Install dependencies — jalankan sekali, lalu restart kernel
# Jika error "No module named 'bertopic'", jalankan cell ini dulu

# !pip install bertopic sentence-transformers

# Jika pip install gagal karena Python 3.13+, gunakan Google Colab:
# https://colab.research.google.com/
print("Uncomment baris !pip install di atas lalu jalankan, kemudian restart kernel.")

Uncomment baris !pip install di atas lalu jalankan, kemudian restart kernel.


In [5]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import os

print("✅ Semua library berhasil diimport!")
print(f"   pandas              : {pd.__version__}")
print(f"   numpy               : {np.__version__}")
print(f"   bertopic            : ok")
print(f"   sentence_transformers: ok")

ModuleNotFoundError: No module named 'bertopic'

## ⚙️ Konfigurasi

Parameter utama BERTopic dan embedding model.

### Parameter BERTopic:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| `EMBEDDING_MODEL` | `paraphrase-multilingual-MiniLM-L12-v2` | Model multilingual ~90MB, support Bahasa Indonesia |
| `N_TOPICS` | `4` | Target jumlah topik utama (sesuai PIPELINE_LPDP_NLP.md) |
| `MIN_TOPIC_SIZE` | `10` | Minimum artikel per topik sebelum dijadikan outlier (-1) |
| `CALCULATE_PROBS` | `True` | Hitung probabilitas keanggotaan topik per dokumen |

> 💡 **Catatan**: Jika hasil 4 topik kurang representatif, coba ubah `N_TOPICS` ke `"auto"` agar BERTopic
> menentukan jumlah topik optimal secara otomatis.

In [ ]:
INPUT_FILE      = "dataset_lpdp_konten_raw.csv"
OUTPUT_FILE     = "dataset_lpdp_topics.csv"
VIZ_FILE        = "topic_sentiment_distribution.png"

# === Konfigurasi BERTopic ===
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
N_TOPICS        = 4     # Target jumlah topik utama
MIN_TOPIC_SIZE  = 10    # Minimum artikel per topik
CALCULATE_PROBS = True

print("⚙️  Konfigurasi berhasil dimuat!")
print(f"   Input file      : {INPUT_FILE}")
print(f"   Output file     : {OUTPUT_FILE}")
print(f"   Embedding model : {EMBEDDING_MODEL}")
print(f"   Target topics   : {N_TOPICS}")
print(f"   Min topic size  : {MIN_TOPIC_SIZE}")

## 📂 Load Dataset

Dataset diambil dari `dataset_lpdp_konten_raw.csv` — output Phase 2 yang berisi **1.038 artikel**
valid dengan konten lengkap hasil scraping (75,8% dari 1.370 artikel valid).

> 📌 Hanya artikel yang memiliki `Content` (tidak null) yang diproses ke BERTopic.
> Artikel tanpa konten akan dikecualikan.

In [ ]:
df = pd.read_csv(INPUT_FILE)
print(f"📂 Source: {INPUT_FILE}")
print(f"   Total rows : {len(df)}")
print(f"   Kolom      : {df.columns.tolist()}")
print()

# === Filter artikel yang punya konten ===
df = df[df['Content'].notna()].copy().reset_index(drop=True)
print(f"✅ Artikel dengan Content : {len(df)}")
print()

# === Distribusi sentimen ===
print(f"{'='*50}")
print("📊 DISTRIBUSI SENTIMEN")
print(f"{'='*50}")
print(f"  {'Label':<12} {'Jumlah':>6}  {'%':>6}")
print(f"  {'-'*28}")
for label in ["Positive", "Neutral", "Negative"]:
    n = (df['Sentiment'] == label).sum()
    print(f"  {label:<12} {n:>6}  {n/len(df)*100:>5.1f}%")
print(f"  {'-'*28}")
print(f"  {'Total':<12} {len(df):>6}  {'100.0%':>6}")
print(f"{'='*50}")

## 🧹 Phase 1 — Preprocessing Teks

Normalisasi teks **minimal** sebelum encoding ke embeddings.

> ⚠️ **Catatan Penting**: BERTopic bekerja lebih baik dengan teks yang **natural** (tidak terlalu
> agresif di-clean). Hindari stemming atau stopword removal di tahap ini — BERTopic memanfaatkan
> konteks semantik yang hilang jika teks terlalu di-strip.

### Langkah Preprocessing:
| Langkah | Aksi | Alasan |
|---------|------|--------|
| Lowercase | `text.lower()` | Normalisasi case |
| Hapus URL | `re.sub(r'http\S+', '', text)` | URL tidak bermakna semantik |
| Hapus whitespace berlebih | `re.sub(r'\s+', ' ', text)` | Normalisasi spasi |
| Trim | `text.strip()` | Hapus leading/trailing space |

In [ ]:
def preprocess_text(text):
    """Normalisasi teks minimal untuk BERTopic — pertahankan konteks semantik."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\S+', '', text)   # hapus URL
    text = re.sub(r'\s+', ' ', text)              # normalisasi whitespace
    return text

docs = df['Content'].apply(preprocess_text).tolist()

print(f"✅ Preprocessing selesai — {len(docs)} dokumen")
print()

# === Statistik panjang dokumen ===
lens = [len(d.split()) for d in docs]
print(f"📊 Statistik Panjang Dokumen (kata):")
print(f"   Min    : {min(lens):,}")
print(f"   Max    : {max(lens):,}")
print(f"   Mean   : {int(np.mean(lens)):,}")
print(f"   Median : {int(np.median(lens)):,}")
print()
print(f"Contoh (idx 0, 200 karakter pertama):")
print(f"  {docs[0][:200]}...")

## 🧬 Phase 2 — Sentence Embeddings

Encode semua dokumen ke vektor dense menggunakan model multilingual.

### Konfigurasi:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| Model | `paraphrase-multilingual-MiniLM-L12-v2` | Support 50+ bahasa termasuk Indonesia |
| Ukuran model | ~90 MB | Download otomatis dari HuggingFace Hub |
| Dimensi vektor | 384 | Output embedding per dokumen |
| Batch size | 32 | Sesuaikan jika RAM terbatas |

> ⏱️ **Estimasi waktu**: ~2–10 menit tergantung hardware (CPU/GPU) dan apakah model sudah di-cache.

In [ ]:
# TODO: [Celine] Jalankan cell ini — download model ~90MB jika belum ada di cache
print(f"⏳ Loading embedding model: {EMBEDDING_MODEL}")
print(f"   (Download otomatis dari HuggingFace Hub jika belum ter-cache)")
print()

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("✅ Model berhasil dimuat!")
print()

# === Encode semua dokumen ===
print(f"⏳ Encoding {len(docs)} dokumen ke vektor...")
print(f"   Estimasi: ~2–10 menit tergantung hardware")

embeddings = embedding_model.encode(
    docs,
    show_progress_bar=True,
    batch_size=32,
)

print(f"\n✅ Embeddings selesai!")
print(f"   Shape : {embeddings.shape}  (n_docs × embedding_dim)")
print(f"   Dtype : {embeddings.dtype}")

## 🗂️ Phase 3 — BERTopic Training

Fit BERTopic menggunakan embeddings yang sudah di-compute di Phase 2.

### Pipeline Internal BERTopic:
| Step | Algoritma | Keterangan |
|------|-----------|------------|
| Dimensionality Reduction | UMAP | Reduksi 384-dim → 5-dim |
| Clustering | HDBSCAN | Clustering berbasis density |
| Topic Representation | c-TF-IDF | Kata representatif per cluster |
| Topic Reduction | `nr_topics=4` | Merge topik kecil → 4 topik utama |

> ⏱️ **Estimasi waktu**: ~2–5 menit  
> ⚠️ **Outliers**: Artikel yang tidak masuk topik manapun diberi label `Topic = -1`.

In [ ]:
# TODO: [Celine] Tune N_TOPICS jika hasil kurang memuaskan (coba "auto" untuk deteksi otomatis)
print(f"⏳ Training BERTopic (target {N_TOPICS} topik)...")
print(f"   Min topic size : {MIN_TOPIC_SIZE}")
print()

topic_model = BERTopic(
    embedding_model=embedding_model,
    nr_topics=N_TOPICS,
    min_topic_size=MIN_TOPIC_SIZE,
    language="multilingual",
    calculate_probabilities=CALCULATE_PROBS,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs, embeddings)

# === Assign hasil ke DataFrame ===
df['Topic']      = topics
df['Topic_Prob'] = probs.max(axis=1) if CALCULATE_PROBS else 0.0

print(f"\n✅ BERTopic training selesai!")
print(f"\n📊 Topic Info:")
topic_info = topic_model.get_topic_info()
print(topic_info.to_string(index=False))

## 🔎 Phase 4 — Analisis & Interpretasi Topik

Interpretasi manual label topik berdasarkan **top words** yang dihasilkan BERTopic,
kemudian analisis distribusi sentimen per topik.

> 📌 **TODO (Celine)**: Setelah melihat top words di output cell berikut, isi `topic_labels`
> dengan nama yang representatif untuk setiap topik.

In [ ]:
# === Top words per topik ===
print("="*60)
print("📋 TOP WORDS PER TOPIK")
print("="*60)
for topic_id in sorted(df['Topic'].unique()):
    if topic_id == -1:
        print(f"\n[Topic -1] OUTLIER — {(df['Topic'] == -1).sum()} artikel (tidak masuk topik manapun)")
        continue
    words  = topic_model.get_topic(topic_id)
    top5   = [w for w, _ in words[:5]]
    top10  = [w for w, _ in words[:10]]
    n      = (df['Topic'] == topic_id).sum()
    print(f"\n[Topic {topic_id}] {n} artikel")
    print(f"  Top 5  : {', '.join(top5)}")
    print(f"  Top 10 : {', '.join(top10)}")

print()

# TODO: [Celine] Isi label manual setelah melihat top words di atas
# Contoh:
# topic_labels = {
#     0: "Kebijakan & Program LPDP",
#     1: "Kisah Awardee & Alumni",
#     2: "Seleksi & Pendaftaran",
#     3: "Polemik & Kontroversi",
# }
topic_labels = {}   # ← Ganti dict kosong ini

if topic_labels:
    df['Topic_Label'] = df['Topic'].map(topic_labels).fillna("Outlier")
    print("✅ Topic labels berhasil di-assign ke kolom 'Topic_Label'")
else:
    df['Topic_Label'] = df['Topic'].apply(lambda x: f"Topic {x}" if x != -1 else "Outlier")
    print("⚠️  topic_labels belum diisi — menggunakan label default 'Topic 0', 'Topic 1', ...")

print()

# === Distribusi topik × sentimen ===
print("="*60)
print("📊 DISTRIBUSI TOPIK × SENTIMEN")
print("="*60)
cross = pd.crosstab(df['Topic_Label'], df['Sentiment'])
print(cross)

## 📊 Phase 5 — Visualisasi

Visualisasi distribusi topik dan hubungannya dengan sentimen.

### Plot yang Dihasilkan:
| Plot | Deskripsi |
|------|-----------|
| Bar chart | Jumlah artikel per topik |
| Heatmap | Proporsi sentimen (%) dalam setiap topik |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("BERTopic — Distribusi Topik Artikel LPDP", fontsize=14, fontweight='bold', y=1.01)

# --- Bar chart: distribusi artikel per topik ---
topic_counts = df['Topic_Label'].value_counts().sort_index()
bars = axes[0].bar(topic_counts.index, topic_counts.values,
                   color='steelblue', edgecolor='white', linewidth=0.8)
axes[0].set_title("Jumlah Artikel per Topik", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Topik")
axes[0].set_ylabel("Jumlah Artikel")
axes[0].tick_params(axis='x', rotation=15)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

# --- Heatmap: proporsi sentimen per topik ---
cross_pct = pd.crosstab(df['Topic_Label'], df['Sentiment'], normalize='index') * 100
# Pastikan urutan kolom konsisten
for col in ["Positive", "Neutral", "Negative"]:
    if col not in cross_pct.columns:
        cross_pct[col] = 0.0
cross_pct = cross_pct[["Positive", "Neutral", "Negative"]]

sns.heatmap(
    cross_pct, annot=True, fmt='.1f', cmap='YlOrRd',
    ax=axes[1], linewidths=0.5, cbar_kws={'label': '%'},
)
axes[1].set_title("Proporsi Sentimen per Topik (%)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Sentimen")
axes[1].set_ylabel("Topik")

plt.tight_layout()
plt.savefig(VIZ_FILE, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Visualisasi disimpan ke: {VIZ_FILE}")

## 💾 Phase 6 — Export Dataset

Simpan dataset dengan topic assignment ke `dataset_lpdp_topics.csv` untuk digunakan di
**Phase 4 — Preprocessing** dan fase berikutnya.

### Kolom Output:
| Kolom | Tipe | Deskripsi |
|-------|------|-----------|
| `Title` | str | Judul artikel |
| `Release Date` | str | Tanggal rilis |
| `URL` | str | Google News URL |
| `Publisher` | str | Nama media |
| `Sentiment` | str | Label manual: Positive / Neutral / Negative |
| `Content` | str | Isi lengkap artikel |
| `Topic` | int | ID topik hasil BERTopic (-1 = outlier) |
| `Topic_Label` | str | Nama topik (hasil interpretasi manual) |
| `Topic_Prob` | float | Probabilitas keanggotaan topik |

In [ ]:
export_cols = [
    'Title', 'Release Date', 'URL', 'Publisher', 'PiC',
    'Valid?', 'Sentiment', 'Notes', 'Actual_URL', 'Content',
    'Topic', 'Topic_Label', 'Topic_Prob',
]

export_df = df[[c for c in export_cols if c in df.columns]].copy()
export_df.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Dataset berhasil disimpan ke: {OUTPUT_FILE}")
print(f"\n📋 Info File:")
print(f"   Rows   : {len(export_df):,}")
print(f"   Columns: {len(export_df.columns)}")
print(f"   Kolom  : {', '.join(export_df.columns.tolist())}")
print(f"   Ukuran : ~{export_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB (in-memory)")
print(f"\n📂 File output: {os.path.abspath(OUTPUT_FILE)}")

## ✅ Summary & Pipeline Checklist

Ringkasan akhir hasil topic discovery artikel LPDP.

In [ ]:
n_total   = len(df)
n_topics  = len([t for t in df['Topic'].unique() if t != -1])
n_outlier = (df['Topic'] == -1).sum()
n_labeled = n_total - n_outlier

print("="*60)
print("🎉 PIPELINE PHASE 3 — SUMMARY AKHIR")
print("="*60)

print(f"\n📌 Proses yang Dilakukan:")
print(f"   1. Load dataset       ✅  {n_total} artikel dengan konten")
print(f"   2. Preprocessing      ✅  Normalisasi teks minimal")
print(f"   3. Embeddings         ✅  {EMBEDDING_MODEL}")
print(f"   4. BERTopic training  ✅  {n_topics} topik ditemukan")
print(f"   5. Analisis topik     ✅  Top words + distribusi per sentimen")
print(f"   6. Visualisasi        ✅  {VIZ_FILE}")
print(f"   7. Export             ✅  {OUTPUT_FILE} ({len(export_df)} artikel)")

print(f"\n📊 Distribusi Topik:")
for label in sorted(df['Topic_Label'].unique()):
    n = (df['Topic_Label'] == label).sum()
    print(f"   {label:<30}: {n:>4} artikel ({n/n_total*100:.1f}%)")

print(f"\n{'-'*60}")
print("✅ CHECKLIST PHASE 3 (PIPELINE_LPDP_NLP.md)")
print(f"{'-'*60}")

topic_labels_defined = bool(topic_labels)
checks = [
    ("Load dataset_lpdp_konten_raw.csv",       True),
    ("Preprocessing teks minimal",              True),
    ("Sentence embeddings encoded",             True),
    (f"BERTopic fit → {n_topics} topik",       True),
    ("Top words per topik dianalisis",          True),
    ("Topic labels didefinisikan manual",       topic_labels_defined),
    ("Visualisasi distribusi topik",            os.path.exists(VIZ_FILE)),
    (f"Export {OUTPUT_FILE}",                   os.path.exists(OUTPUT_FILE)),
]

all_passed = True
for item, done in checks:
    status = "✅" if done else "⬜"
    if not done:
        all_passed = False
    print(f"  {status} {item}")

print(f"\n{'='*60}")
if all_passed:
    print(f"🎊 Phase 3 SELESAI! {n_labeled} artikel ter-assign ke topik, siap Phase 4.")
else:
    print("⚠️  Ada item belum selesai — cek ⬜ di atas.")
    if not topic_labels_defined:
        print("   → Isi 'topic_labels' dict di Phase 4 setelah interpretasi top words.")
print("="*60)

print(f"\n➡️  Next Steps:")
print(f"   Phase 4 — Preprocessing NLP pipeline 10 langkah (PIC: Iqbal)")
print(f"   Input: {OUTPUT_FILE}")